### Dataset curado con splits fijos;ventaneo HU funcionando, análisis exploratorio (EDA) de fragmentos por caso; visualizador 1 (MIP raw) implementado y operativo.

In [2]:
%pip install scikit-learn scikit-image

  Using cached scipy-1.18.1-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB 2.0 MB/s eta 0:00:05
   ---------------------------------------- 0.1/8.3 MB 1.1 MB/s eta 0:00:08
    --------------------------------------- 0.2/8.3 MB 1.5 MB/s eta 0:00:06
   -- ------------------------------------- 0.4/8.3 MB 2.4 MB/s eta 0:00:04
   -- ------------------------------------- 0.6/8.3 MB 2.6 MB/s eta 0:00:03
   ---- ----------------------------------- 1.0/8.3 MB 3.9 MB/s eta 0:00:02
   --------- ------------------------------ 2.0/8.3 MB 6.4 MB/s eta 0:00:01
   ------------- -------------------------- 2.9/8.3 MB 7.9 MB/s eta 0:00:01
   -------------------- ------------------- 4.3/8.3 MB 10.6 MB/s eta 0:00:01
   ---------------------------------- ----- 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import random
from pathlib import Path
 
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from skimage.filters import threshold_otsu
from skimage.exposure import equalize_adapthist, adjust_gamma

In [5]:
BASE_DIR = Path.cwd()

DATA_MHA_DIR = BASE_DIR / "data_mha"

IMAGE_DIRS = [
    DATA_MHA_DIR / "PENGWIN_CT_train_images_part1",
    DATA_MHA_DIR / "PENGWIN_CT_train_images_part2",
]

LABELS_DIR = DATA_MHA_DIR / "PENGWIN_CT_train_labels"
LABEL_DIR = LABELS_DIR  # Alias para compatibilidad con celdas anteriores
SPLITS_DIR = BASE_DIR / "splits"
SPLITS_DIR.mkdir(exist_ok=True)


In [6]:
image_files = []

for image_dir in IMAGE_DIRS:
    image_files.extend(image_dir.glob("*.mha"))

image_files = sorted(image_files)
label_files = sorted(LABELS_DIR.glob("*.mha"))

print("RESUMEN DEL DATASET")


print(f"Imágenes encontradas: {len(image_files)}")
print(f"Labels encontrados:   {len(label_files)}")

RESUMEN DEL DATASET
Imágenes encontradas: 100
Labels encontrados:   100


In [7]:
image_ids = {
    file.stem
    for file in image_files
}

label_ids = {
    file.stem
    for file in label_files
}

In [8]:
images_without_labels = sorted(image_ids - label_ids)
labels_without_images = sorted(label_ids - image_ids)

common_ids = sorted(image_ids & label_ids)

print("VALIDACIÓN DE CORRESPONDENCIA")

print(f"Casos con imagen + label: {len(common_ids)}")

print(
    f"Imágenes sin label:       {len(images_without_labels)}"
)

print(
    f"Labels sin imagen:        {len(labels_without_images)}"
)


if images_without_labels:
    print("\nImágenes sin label:")
    for case_id in images_without_labels:
        print(f"  - {case_id}")


if labels_without_images:
    print("\nLabels sin imagen:")
    for case_id in labels_without_images:
        print(f"  - {case_id}")


VALIDACIÓN DE CORRESPONDENCIA
Casos con imagen + label: 100
Imágenes sin label:       0
Labels sin imagen:        0


In [9]:
#emparejar imagenes y labels

def build_dataset_index(image_dirs, labels_dir):
    image_files = []
    for image_dir in image_dirs:
        image_files.extend(image_dir.glob("*.mha"))
    image_files = sorted(image_files)
    label_files = sorted(labels_dir.glob("*.mha"))
 
    image_ids = {f.stem for f in image_files}
    label_ids = {f.stem for f in label_files}
 
    images_without_labels = sorted(image_ids - label_ids)
    labels_without_images = sorted(label_ids - image_ids)
    common_ids = sorted(image_ids & label_ids)
 
    print("VALIDACIÓN DE CORRESPONDENCIA")
    print(f"Casos con imagen + label: {len(common_ids)}")
    print(f"Imágenes sin label:       {len(images_without_labels)}")
    print(f"Labels sin imagen:        {len(labels_without_images)}")
    if images_without_labels:
        print("  Imágenes sin label:", images_without_labels)
    if labels_without_images:
        print("  Labels sin imagen:", labels_without_images)
 
    dataset = []
    for case_id in common_ids:
        image_path = None
        for image_dir in image_dirs:
            possible_path = image_dir / f"{case_id}.mha"
            if possible_path.exists():
                image_path = possible_path
                break
        label_path = labels_dir / f"{case_id}.mha"
        if image_path is not None and label_path.exists():
            dataset.append({
                "case_id": case_id,
                "image_path": str(image_path),
                "label_path": str(label_path),
            })
 
    return pd.DataFrame(dataset)

In [10]:
def load_mha(path):
    """
    Carga un volumen .mha con SimpleITK.
    Devuelve:
        volume: np.array con orden de ejes (Z, Y, X)
        spacing: tupla (x_mm, y_mm, z_mm)
    """
    img = sitk.ReadImage(str(path))
    volume = sitk.GetArrayFromImage(img)   # (Z, Y, X)
    spacing = img.GetSpacing()             # (X, Y, Z) en mm
    return volume, spacing

In [13]:
#CONTEO DE FRAGMENTOS POR CASO (para poder estratificar el split)

def compute_fragment_counts(df):
    frag_counts = []
    for _, row in df.iterrows():
        label_vol, _ = load_mha(row["label_path"])
        n_fragments = len(np.unique(label_vol)) - 1  # -1 por excluir el fondo
        frag_counts.append(n_fragments)
    df = df.copy()
    df["n_fragments"] = frag_counts
    return df

In [ ]:
from sklearn.model_selection import train_test_split

# Bins según tu distribución real (3-9 fragmentos)
df["frag_bin"] = pd.cut(
    df["n_fragments"],
    bins=[0, 4, 6, 100],
    labels=["bajo", "medio", "alto"]
)

print(df["frag_bin"].value_counts())

# Primer split: 75% train vs 25% restante
train_df, temp_df = train_test_split(
    df, test_size=0.25, stratify=df["frag_bin"], random_state=42
)

# Segundo split: del 25% restante, separar test (20% del total) y val (5% del total)
# proporción dentro de temp_df: val = 5/25 = 0.20
test_df, val_df = train_test_split(
    temp_df, test_size=0.20, stratify=temp_df["frag_bin"], random_state=42
)

print(f"Train: {len(train_df)}, Test: {len(test_df)}, Val: {len(val_df)}")

SPLITS_DIR.mkdir(exist_ok=True)
train_df["case_id"].to_csv(SPLITS_DIR / "train.txt", index=False, header=False)
test_df["case_id"].to_csv(SPLITS_DIR / "test.txt", index=False, header=False)
val_df["case_id"].to_csv(SPLITS_DIR / "val.txt", index=False, header=False)